---
title: "Generate Cards for Group Member Listing"
jupyter: "card-lab"
execute: 
  enabled: true
format: 
    html: 
        default: false
---

In [1]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Load Excel sheet
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

filename = "private/CARD Group Timeline.xlsx"

df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

names = df['Display Name'].values

# Get the current group members and alumni
finish_series = df['Ultimate Role Finish']
finish_text = finish_series.astype(str).str.strip().str.lower()
current_mask = finish_series.isna() | finish_text.isin(['', 'nan', 'nat']) | finish_text.str.contains('current', na=False)
alumni_mask = ~current_mask

current_group_member_indices = df[current_mask].index.tolist()
alumni_indices = df[alumni_mask].index.tolist()

current_group_members = df.loc[current_group_member_indices, 'Display Name']
alumni = df.loc[alumni_indices, 'Display Name']

current_index = np.zeros([len(names), 1])
current_index[current_group_member_indices] = 1
alumni_index = np.zeros([len(names), 1])
alumni_index[alumni_indices] = 1
df['current'] = current_index
df['alumni'] = alumni_index
del current_index, alumni_index

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



In [ ]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# <!-- Future TODO: Store geocached data -->

import requests
import json
import shutil
from pathlib import Path
from PIL import Image, ImageOps

ZOTERO_GROUP_ID = '5985739'
LOCAL_ZOTERO_ITEMS_PATH = Path('./files/zotero-items.json')
HEADSHOT_SOURCE_NOTEBOOK_PATH = Path('./_people-action.ipynb')

# Listing cards use 250px image height; keep 2x pixel density for crisp web output.
HEADSHOT_DISPLAY_SIZE_PX = 250
HEADSHOT_RETINA_SCALE = 2
HEADSHOT_MAX_DIM_PX = HEADSHOT_DISPLAY_SIZE_PX * HEADSHOT_RETINA_SCALE
HEADSHOT_WEB_DPI = 96
HEADSHOT_CROP_CENTER_X = 0.5
HEADSHOT_CROP_CENTER_Y = 0.44
HEADSHOT_ENABLE_FACE_DETECTION = True
HEADSHOT_FACE_HEIGHT_TARGET_RATIO = 0.32
HEADSHOT_FACE_HEIGHT_MIN_RATIO = 0.36
HEADSHOT_FALLBACK_CROP_SCALE = 0.9
HEADSHOT_MIN_CROP_SCALE_WITH_FACE = 0.90
_LOCAL_ZOTERO_ITEMS_CACHE = None

def country_code_to_unicode_codepoints(country_code):
#def country_code_to_html_entities(country_code):
    """
    Convert an ISO 3166-1 alpha-2 country code to HTML character entities for flag emoji.
HEADSHOT_PERSON_CROP_MODES = {
    'Ravi_Kumar': 'pad',
    'Ian_Giblin': 'pad',
    'Phillisity_Neal': 'pad',
    'Nick_Jackson': 'pad',
    'Logan_Herr': 'pad',
}
    
    Args:
        country_code (str): Two-letter country code, e.g., 'US', 'FR'
    
    Returns:
        str: HTML character entity string, e.g., '&#127482;&#127480;'
    """
    if not isinstance(country_code, str) or len(country_code) != 2 or not country_code.isalpha():
        raise ValueError("Input must be a 2-letter alphabetic ISO-3166-1 country code.")

    country_code = country_code.upper()
    entities = [
        f"&#{ord(c) - ord('A') + 0x1F1E6};"
        for c in country_code
    ]
    return ''.join(entities)

def get_wikipedia_url_from_country_code(iso_code):
    """
    Given an ISO 3166-1 alpha-2 country code, return the Wikipedia article URL for that country.

    Args:
        iso_code (str): Two-letter ISO country code (e.g., 'US', 'FR', 'JP').

    Returns:
        str: Wikipedia article URL, or None if code is invalid.
    """
    import pycountry
    import urllib.parse
    try:
        country = pycountry.countries.get(alpha_2=iso_code.upper())
        if country is None:
            return None
        country_name = country.name

        # Some countries have "official" long names that are more accurate for Wikipedia:
        # Try to use the "official_name" if available (e.g., for "Korea, Republic of")
        if hasattr(country, "official_name"):
            country_name = country.official_name

        # Encode for URL
        article_title = urllib.parse.quote(country_name.replace(" ", "_"))
        return f"https://en.wikipedia.org/wiki/{article_title}"

    except Exception as e:
        print(f"Error: {e}")
        return None

import urllib.parse

# Mapping from U.S. state abbreviation to full state name
US_STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming",
    # Optionally add territories
    "DC": "District of Columbia", "PR": "Puerto Rico", "GU": "Guam", "VI": "United States Virgin Islands",
}

def get_us_state_wikipedia_url(abbreviation):
    """
    Given a two-letter U.S. state abbreviation, return the Wikipedia article URL.

    Args:
        abbreviation (str): e.g. 'CA', 'NY', 'TX'

    Returns:
        str: Wikipedia URL string, or None if abbreviation is invalid.
    """
    abbreviation = abbreviation.upper()
    state_name = US_STATE_NAMES.get(abbreviation)
    if not state_name:
        return None

    # Replace spaces with underscores and encode special characters
    article_title = urllib.parse.quote(state_name.replace(" ", "_"))
    return f"https://en.wikipedia.org/wiki/{article_title}"

def get_country_iso_code_nominatim(address, user_agent="country-lookup-script"):
    """
    Uses Nominatim (OpenStreetMap) to geocode an address and return the ISO 3166-1 alpha-2 country code.
    No API key required, but be respectful of usage limits.
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        'q': address,
        'format': 'json',
        'addressdetails': 1,
        'limit': 1,
    }
    headers = {
        'User-Agent': user_agent
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()

        if data and 'address' in data[0]:
            return data[0]['address'].get('country_code', '').upper()
        else:
            return None
    except Exception as e:
        print(f"Error during geocoding: {e}")
        return None

def get_country_flag_icon_url(country_code, format='svg'):
    """
    Returns the URL of the country flag icon from FlagCDN for a given ISO 3166-1 alpha-2 country code.
    
    Parameters:
        country_code (str): The two-letter country code, e.g., 'US', 'DE', 'FR'.
        format (str): 'svg' (default) or 'png'.
        
    Returns:
        str: URL to the flag image.
    """
    country_code = country_code.lower()
    if format == 'svg':
        return f"https://flagcdn.com/{country_code}.svg"
    elif format == 'png':
        return f"https://flagcdn.com/w80/{country_code}.png"
    else:
        raise ValueError("Format must be 'svg' or 'png'")

def get_us_state_flag_icon_url(us_state_code, format='svg'):
    """
    Returns the URL of the country flag icon from FlagCDN for a given ISO 3166-1 alpha-2 country code.
    
    Parameters:
        us_state_code (str): The two-letter country code, e.g., 'US', 'DE', 'FR'.
        format (str): 'svg' (default) or 'png'.
        
    Returns:
        str: URL to the flag image.
    """
    us_state_code = us_state_code.lower()
    if format == 'svg':
        return f"https://flagcdn.com/us-{us_state_code}.svg"
    elif format == 'png':
        return f"https://flagcdn.com/w80/us-{us_state_code}.png"
    else:
        raise ValueError("Format must be 'svg' or 'png'")

def get_us_state_code(address_string):
    import usaddress
    """
    Extracts the US state code (two-letter abbreviation) from an address string.

    Args:
        address_string (str): The full address string.

    Returns:
        str or None: The two-letter state code if found, otherwise None.
    """
    try:
        # Parse the address string
        parsed_address = usaddress.parse(address_string)

        # Iterate through the parsed components to find the state
        for component, label in parsed_address:
            if label == 'StateName':
                return component.upper()  # Return the state name in uppercase (e.g., 'CA')
        return None  # State not found
    except usaddress.RepeatedLabelError as e:
        print(f"Error parsing address: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

def load_local_zotero_items(path=LOCAL_ZOTERO_ITEMS_PATH):
    global _LOCAL_ZOTERO_ITEMS_CACHE
    if _LOCAL_ZOTERO_ITEMS_CACHE is not None:
        return _LOCAL_ZOTERO_ITEMS_CACHE

    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list):
            _LOCAL_ZOTERO_ITEMS_CACHE = data
            return _LOCAL_ZOTERO_ITEMS_CACHE
    except Exception as e:
        print(f"Warning: failed to load local Zotero items from {path}: {e}")

    _LOCAL_ZOTERO_ITEMS_CACHE = []
    return _LOCAL_ZOTERO_ITEMS_CACHE

def _clamp(value, lower=0.0, upper=1.0):
    return max(lower, min(upper, value))

def _pad_to_square_no_zoom(img):
    if img.width == img.height:
        return img

    square_side = max(img.width, img.height)
    padded = Image.new('RGB', (square_side, square_side), (255, 255, 255))
    source = img.convert('RGB') if img.mode != 'RGB' else img
    left = (square_side - source.width) // 2
    top = (square_side - source.height) // 2
    padded.paste(source, (left, top))
    return padded

def _detect_face_box_opencv(img):
    if not HEADSHOT_ENABLE_FACE_DETECTION:
        return None

    try:
        import cv2
    except Exception:
        return None

    try:
        rgb = np.array(img.convert('RGB'))
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        cascade_dir = getattr(getattr(cv2, 'data', None), 'haarcascades', '')
        if not cascade_dir:
            return None

        cascade_path = Path(cascade_dir) / 'haarcascade_frontalface_default.xml'
        detector = cv2.CascadeClassifier(str(cascade_path))
        if detector.empty():
            return None

        min_side = max(48, min(gray.shape[:2]) // 8)
        faces = detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(min_side, min_side),
        )
        if len(faces) == 0:
            return None

        x, y, width, height = max(faces, key=lambda face: face[2] * face[3])
        return (int(x), int(y), int(width), int(height))
    except Exception:
        return None

def _get_headshot_crop_box(img):
    min_side = min(img.width, img.height)
    if min_side <= 0:
        return (0, 0, img.width, img.height)

    crop_side = min_side
    face_box = _detect_face_box_opencv(img)
    if face_box:
        x, y, width, height = face_box
        face_center_x = x + (width / 2.0)
        face_eye_y = y + (height * 0.42)

        target_side = height / HEADSHOT_FACE_HEIGHT_TARGET_RATIO
        max_side_from_face = height / HEADSHOT_FACE_HEIGHT_MIN_RATIO
        crop_side = int(round(max(height + 2, min(target_side, max_side_from_face))))
        crop_side = max(1, min(crop_side, min_side))

        left = round(face_center_x - (crop_side / 2.0))
        top = round(face_eye_y - (crop_side * HEADSHOT_CROP_CENTER_Y))
    else:
        crop_side = max(1, int(round(min_side * HEADSHOT_FALLBACK_CROP_SCALE)))
        left = round((img.width - crop_side) * HEADSHOT_CROP_CENTER_X)
        top = round((img.height - crop_side) * HEADSHOT_CROP_CENTER_Y)

    left = int(max(0, min(left, img.width - crop_side)))
    top = int(max(0, min(top, img.height - crop_side)))
    return (left, top, left + crop_side, top + crop_side)

def copy_downsampled_headshot(private_path, public_path):
    src = Path(private_path)
    dst = Path(public_path)
    dst.parent.mkdir(parents=True, exist_ok=True)

    try:
        with Image.open(src) as img:
            exif = img.getexif()
            img = ImageOps.exif_transpose(img)

            try:
                del exif[274]
            except Exception:
                pass

            crop_mode = HEADSHOT_PERSON_CROP_MODES.get(src.stem, 'face')
            face_box = None
            if crop_mode != 'pad':
                face_box = _detect_face_box_opencv(img)

            if crop_mode == 'pad' or face_box is None:
                img = _pad_to_square_no_zoom(img)
            else:
                crop_box = _get_headshot_crop_box(img, face_box=face_box)
                img = img.crop(crop_box)

            if img.width > HEADSHOT_MAX_DIM_PX or img.height > HEADSHOT_MAX_DIM_PX:
                img.thumbnail((HEADSHOT_MAX_DIM_PX, HEADSHOT_MAX_DIM_PX), Image.Resampling.LANCZOS)

            suffix = dst.suffix.lower()
            save_kwargs = {'dpi': (HEADSHOT_WEB_DPI, HEADSHOT_WEB_DPI)}

            if suffix in ('.jpg', '.jpeg'):
                if img.mode not in ('RGB', 'L'):
                    img = img.convert('RGB')
                save_kwargs.update({'quality': 85, 'optimize': True, 'progressive': True, 'exif': exif.tobytes()})
            elif suffix == '.png':
                save_kwargs.update({'optimize': True, 'compress_level': 9})

            img.save(dst, **save_kwargs)
    except Exception as e:
        shutil.copy2(src, dst)
        print(f"Warning: copied without downsampling for {src.name}: {e}")

def get_zotero_items_by_author_and_type(author_name, allowed_types):
    from pyzotero import zotero
    zot = zotero.Zotero(ZOTERO_GROUP_ID, 'group')

    search_initials = ''
    try:
        search_last, search_initials = author_name.split(",")
        search_last = search_last.strip()
        search_initials = search_initials.strip()
    except:
        search_last = author_name

    local_items = load_local_zotero_items()
    if local_items:
        items = [
            item for item in local_items
            if search_last.lower() in str(item.get('meta', {}).get('creatorSummary', '')).lower()
            or search_last.lower() in str(item.get('data', {}).get('title', '')).lower()
            or any(search_last.lower() in str(c.get('lastName', '')).lower() for c in item.get('data', {}).get('creators', []))
        ]
    else:
        items = []

    if len(search_initials) > 0:
        filtered_items = []
        for item in items:
            data = item['data']
            item_type = data.get('itemType', '')
            creators = data.get('creators', [])

            if item_type in allowed_types:
                if matches_author(creators, search_last, search_initials):
                    filtered_items.append(item)
    else:
        filtered_items = []
        for item in items:
            data = item['data']
            item_type = data.get('itemType', '')
            creators = data.get('creators', [])

            if item_type in allowed_types:
                if any(author_name.lower() in c.get('lastName', '').lower() for c in creators):
                    filtered_items.append(item)

    if filtered_items:
        return filtered_items

    # Backup option: query Zotero group library directly if local cache has no matches.
    try:
        items = zot.items(q=search_last, qmode='everything')
    except Exception as e:
        print(f"Warning: Zotero group fallback failed for {author_name}: {e}")
        return []

    if len(search_initials) > 0:
        filtered_items = []
        for item in items:
            data = item['data']
            item_type = data.get('itemType', '')
            creators = data.get('creators', [])

            if item_type in allowed_types:
                if matches_author(creators, search_last, search_initials):
                    filtered_items.append(item)
    else:
        filtered_items = []
        for item in items:
            data = item['data']
            item_type = data.get('itemType', '')
            creators = data.get('creators', [])

            if item_type in allowed_types:
                if any(author_name.lower() in c.get('lastName', '').lower() for c in creators):
                    filtered_items.append(item)

    return filtered_items

def get_formatted_citations(item_keys, style):
    import requests
    from bs4 import BeautifulSoup, NavigableString

    if not item_keys:
        return ''

    keys_csv = ",".join(item_keys)
    url = f"https://api.zotero.org/groups/{ZOTERO_GROUP_ID}/items"
    headers = {
        'Accept': 'text/html'  # Or text/plain if preferred
    }
    params = {
        'itemKey': keys_csv,
        'format': 'bib',
        'style': style,
        'sort': 'date',
        'direction': 'desc'
    }

    response = requests.get(url, headers=headers, params=params)
    if response.status_code != 200:
        fallback_params = {
            'itemKey': keys_csv,
            'format': 'bib',
            'style': style
        }
        response = requests.get(url, headers=headers, params=fallback_params)

    if response.status_code != 200:
        return ''

    import re

    def _entry_sort_key(entry_text):
        month_map = {
            'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
            'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
        }

        iso_match = re.search(r'(19|20)\d{2}-(\d{2})-(\d{2})', entry_text)
        if iso_match:
            year = int(iso_match.group(0)[0:4])
            month = int(iso_match.group(2))
            day = int(iso_match.group(3))
            return (year, month, day)

        month_year_match = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\.?\s+((?:19|20)\d{2})', entry_text, re.IGNORECASE)
        if month_year_match:
            month = month_map[month_year_match.group(1).lower()]
            year = int(month_year_match.group(2))
            return (year, month, 0)

        year_match = re.search(r'(19|20)\d{2}', entry_text)
        if year_match:
            return (int(year_match.group(0)), 0, 0)

        return (0, 0, 0)

    soup = BeautifulSoup(response.text, 'html.parser')
    bib_body = soup.select_one('div.csl-bib-body')
    if bib_body is None:
        return response.text

    entries = bib_body.find_all('div', class_='csl-entry', recursive=False)
    total_entries = len(entries)

    if total_entries == 0:
        return response.text

    doi_pattern = re.compile(r'(10\.\d{4,9}/[-._;()/:A-Z0-9]+)', re.IGNORECASE)
    url_pattern = re.compile(r'(https?://[^\s<>"]+)', re.IGNORECASE)
    online_available_pattern = re.compile(r'\[Online\]\.\s*Available:\s*', re.IGNORECASE)

    def _linkify_doi_text(entry):
        for text_node in list(entry.find_all(string=True)):
            parent_name = getattr(text_node.parent, 'name', None)
            if parent_name in {'a', 'script', 'style'}:
                continue

            text = str(text_node)
            if '10.' not in text:
                continue

            matches = list(doi_pattern.finditer(text))
            if not matches:
                continue

            last_index = 0
            replacement_nodes = []
            for match in matches:
                start, end = match.span(1)
                if start > last_index:
                    replacement_nodes.append(NavigableString(text[last_index:start]))

                doi = match.group(1)
                link = BeautifulSoup('', 'html.parser').new_tag('a', href=f'https://doi.org/{doi}')
                link.string = doi
                replacement_nodes.append(link)
                last_index = end

            if last_index < len(text):
                replacement_nodes.append(NavigableString(text[last_index:]))

            for replacement_node in reversed(replacement_nodes):
                text_node.insert_after(replacement_node)
            text_node.extract()

    def _linkify_url_text(entry):
        for text_node in list(entry.find_all(string=True)):
            parent_name = getattr(text_node.parent, 'name', None)
            if parent_name in {'a', 'script', 'style'}:
                continue

            text = str(text_node)
            if 'http://' not in text and 'https://' not in text:
                continue

            matches = list(url_pattern.finditer(text))
            if not matches:
                continue

            last_index = 0
            replacement_nodes = []
            for match in matches:
                start, end = match.span(1)
                if start > last_index:
                    replacement_nodes.append(NavigableString(text[last_index:start]))

                url = match.group(1).rstrip('.,;')
                trailing = match.group(1)[len(url):]
                link = BeautifulSoup('', 'html.parser').new_tag('a', href=url)
                link.string = url
                replacement_nodes.append(link)
                if trailing:
                    replacement_nodes.append(NavigableString(trailing))
                last_index = end

            if last_index < len(text):
                replacement_nodes.append(NavigableString(text[last_index:]))

            for replacement_node in reversed(replacement_nodes):
                text_node.insert_after(replacement_node)
            text_node.extract()

    def _strip_online_available_text(entry):
        for text_node in list(entry.find_all(string=True)):
            parent_name = getattr(text_node.parent, 'name', None)
            if parent_name in {'a', 'script', 'style'}:
                continue

            text = str(text_node)
            cleaned = online_available_pattern.sub('', text)
            if cleaned != text:
                text_node.replace_with(NavigableString(cleaned))

    ordered_entries = sorted(
        entries,
        key=lambda entry: _entry_sort_key(entry.get_text(' ', strip=True)),
        reverse=True
    )

    for entry in entries:
        entry.extract()

    for index, entry in enumerate(ordered_entries):
        display_index = total_entries - index
        left_margin = entry.select_one('div.csl-left-margin')
        if left_margin:
            left_margin.clear()
            left_margin.append(f"[{display_index}]")
        _strip_online_available_text(entry)
        _linkify_doi_text(entry)
        _linkify_url_text(entry)
        bib_body.append(entry)

    rendered = str(soup)
    rendered = re.sub(r'^<\?xml[^>]*>\s*', '', rendered)
    return rendered

def _creator_display_name(creator):
    if creator.get('name'):
        return creator.get('name')

    first = creator.get('firstName', '').strip()
    last = creator.get('lastName', '').strip()
    first_parts = [part for part in first.replace('-', ' ').split() if part]
    initials = ' '.join([f"{part[0].upper()}." for part in first_parts])

    if initials and last:
        return f"{initials} {last}"
    if last:
        return last
    if initials:
        return initials

    full = f"{first} {last}".strip()
    return full

def get_presentation_citations_all_authors(items, person_name=None):
    import html
    import re
    from datetime import datetime

    def _presentation_sort_key(raw_date):
        text = str(raw_date or '').strip()
        if not text:
            return (0, 0, 0)

        formats = [
            '%Y-%m-%d', '%Y-%m', '%Y',
            '%B %d, %Y', '%b %d, %Y',
            '%B %Y', '%b %Y',
        ]

        for fmt in formats:
            try:
                parsed = datetime.strptime(text, fmt)
                return (parsed.year, parsed.month, parsed.day)
            except ValueError:
                continue

        year_match = re.search(r'(19|20)\d{2}', text)
        if year_match:
            return (int(year_match.group(0)), 0, 0)

        return (0, 0, 0)

    def _format_entries_list(entries_list):
        if not entries_list:
            return ''

        entries_list.sort(key=lambda pair: pair[0], reverse=True)
        rendered = []
        total_entries = len(entries_list)
        for index, (_, entry_text) in enumerate(entries_list):
            display_index = total_entries - index
            rendered.append(
                '  <div class="csl-entry" style="clear: left; ">\n'
                f'    <div class="csl-left-margin" style="float: left; padding-right: 0.5em; text-align: right; width: 2em;">[{display_index}]</div>'
                f'<div class="csl-right-inline" style="margin: 0 .4em 0 2.5em;">{entry_text}</div>\n'
                '  </div>'
            )

        return '<div class="csl-bib-body" style="line-height: 1.35; ">\n' + '\n'.join(rendered) + '\n</div>'

    presenter_entries = []
    contributor_entries = []

    normalized_person_name = str(person_name or '').strip().lower()

    def _creator_matches_person(creator, display_name):
        target = str(display_name or '').strip()
        if not target:
            return False

        target_parts = [part for part in target.replace('-', ' ').split() if part]
        if not target_parts:
            return False

        target_last = target_parts[-1].lower()
        target_initial = target_parts[0][0].lower()

        creator_last = str(creator.get('lastName', '') or '').strip().lower()
        creator_first = str(creator.get('firstName', '') or '').strip()
        if creator_last:
            if creator_last != target_last:
                return False
            if creator_first:
                return creator_first[0].lower() == target_initial

        parsed_name = _creator_display_name(creator)
        parsed_parts = [part for part in str(parsed_name or '').replace('-', ' ').split() if part]
        if len(parsed_parts) < 2:
            return False

        parsed_last = parsed_parts[-1].lower()
        parsed_first_token = parsed_parts[0].replace('.', '')
        if not parsed_first_token:
            return False

        return parsed_last == target_last and parsed_first_token[0].lower() == target_initial

    for item in items:
        data = item.get('data', {})
        ordered_creators = []
        person_role = None

        for creator in data.get('creators', []):
            display_name = _creator_display_name(creator)
            if not display_name:
                continue

            safe_name = html.escape(display_name)
            creator_type = str(creator.get('creatorType', '')).lower()
            if creator_type == 'presenter':
                ordered_creators.append(f"<em>{safe_name}</em> (presenter)")
            else:
                ordered_creators.append(safe_name)

            if normalized_person_name and _creator_matches_person(creator, person_name):
                person_role = creator_type

        authors_text = ', '.join(ordered_creators) if ordered_creators else 'Unknown author'

        raw_date = data.get('date', 'n.d.')
        date = html.escape(str(raw_date))
        title = html.escape(str(data.get('title', 'Untitled')))

        meeting_name = data.get('meetingName') or data.get('proceedingsTitle') or ''
        meeting_name = html.escape(str(meeting_name))

        if meeting_name:
            entry = f"{authors_text}. ({date}). {title}. <em>{meeting_name}</em>."
        else:
            entry = f"{authors_text}. ({date}). {title}."

        presentation_url = str(data.get("url") or "").strip()
        if presentation_url:
            safe_url = html.escape(presentation_url, quote=True)
            entry = entry.rstrip()
            if entry.endswith('.'):
                entry = entry[:-1]
            entry += f'. <a href="{safe_url}" target="_blank">{safe_url}</a>.'

        bucket = presenter_entries
        if normalized_person_name:
            if person_role == 'presenter':
                bucket = presenter_entries
            else:
                bucket = contributor_entries

        bucket.append((_presentation_sort_key(raw_date), entry))

    presenter_html = _format_entries_list(presenter_entries)
    contributor_html = _format_entries_list(contributor_entries)
    return presenter_html, contributor_html
def get_people_workproducts_assets():
    return """```{=html}
<style>
.citation-badges { display: inline-flex; align-items: center; gap: 6px; margin-left: 6px; }
.citation-badges .altmetric-embed, .citation-badges .__dimensions_badge_embed__ { display: inline-block !important; vertical-align: middle; margin-left: 0; }
.citation-badges .oa-icon-link, .citation-badges .pdf-icon-link { display: inline-flex !important; align-items: center; }
.csl-right-inline .citation-badges, .citation .citation-badges { white-space: nowrap; }
.oa-icon { margin-left: 6px; vertical-align: middle; display: inline-flex; align-items: center; }
.oa-gold { color: #FFD700; } .oa-green { color: #2e7d32; } .oa-bronze { color: #cd7f32; }
.oa-hybrid { color: #ff8f00; } .oa-diamond { color: #b9f2ff; } .oa-unknown { color: #1e88e5; }
.pdf-icon { margin-left: 4px; color: #e00122; vertical-align: middle; display: inline-flex; align-items: center; }
</style>
<script type='text/javascript' src='//d1bxh8uas1mnw7.cloudfront.net/assets/embed.js'></script>
<script async src='https://badge.dimensions.ai/badge.js' charset='utf-8'></script>
<script src='https://code.iconify.design/2/2.2.1/iconify.min.js'></script>
<script>(function(){function c(s){switch(s){case 'gold':return 'oa-gold';case 'green':return 'oa-green';case 'bronze':return 'oa-bronze';case 'hybrid':return 'oa-hybrid';case 'diamond':return 'oa-diamond';default:return 'oa-unknown';}}function d(t){if(!t)return null;const m=t.match(/10\.\d{4,9}\/[-._;()/:A-Z0-9]+/i);return m?m[0].toLowerCase().replace(/[.,;]$/,''):null;}async function e(){const n=[...document.querySelectorAll('.csl-entry,.citation')];const map=new Map();const dois=[];n.forEach((node)=>{let doi=null;for(const a of node.querySelectorAll('a[href]')){const h=a.getAttribute('href')||'';const k=h.includes('doi.org/')?h.split('doi.org/').pop():null;doi=d(k||a.textContent||h);if(doi)break;}if(!doi)doi=d(node.textContent||'');if(doi){map.set(doi,node);dois.push(doi);}});if(!dois.length)return;const chunks=[];for(let i=0;i<dois.length;i+=100)chunks.push(dois.slice(i,i+100));const works=[];for(const p of chunks){const f=p.map((x)=>encodeURIComponent(x)).join('|');try{const r=await fetch(`https://api.openalex.org/works?filter=doi:${f}`);if(!r.ok)continue;const j=await r.json();if(Array.isArray(j.results))works.push(...j.results);}catch(_){}}works.forEach((w)=>{const doi=(w.doi||'').replace(/^https?:\/\/doi.org\//i,'').toLowerCase();const node=map.get(doi);if(!node||node.querySelector('[data-people-citation-badges]'))return;const b=document.createElement('span');b.className='citation-badges';b.setAttribute('data-people-citation-badges','1');const a=document.createElement('span');a.className='altmetric-embed';a.setAttribute('data-badge-type','4');a.setAttribute('data-doi',doi);a.setAttribute('data-hide-no-mentions','true');b.appendChild(a);const dm=document.createElement('span');dm.className='__dimensions_badge_embed__';dm.setAttribute('data-style','small_rectangle');dm.setAttribute('data-doi',doi);dm.setAttribute('data-hide-zero-citations','true');b.appendChild(dm);if(w.open_access&&w.open_access.is_oa&&w.open_access.oa_url){const ol=document.createElement('a');ol.href=w.open_access.oa_url;ol.target='_blank';ol.className='oa-icon-link oa-icon '+c(w.open_access.oa_status||'unknown');const oi=document.createElement('span');oi.className='iconify';oi.setAttribute('data-icon','academicons:open-access');oi.setAttribute('data-width','20');oi.setAttribute('data-height','20');ol.appendChild(oi);b.appendChild(ol);const arr=w.open_access.oa_urls||[];const rx=/\.pdf(\?|$)/i;let pdf=null;if(Array.isArray(arr)){const pe=arr.find((x)=>rx.test((x&&x.url)||''));if(pe)pdf=pe.url;}if(!pdf&&rx.test(w.open_access.oa_url))pdf=w.open_access.oa_url;if(pdf){const pl=document.createElement('a');pl.href=pdf;pl.target='_blank';pl.className='pdf-icon-link pdf-icon';const pi=document.createElement('span');pi.className='iconify';pi.setAttribute('data-icon','mdi:file-pdf-box');pi.setAttribute('data-width','20');pi.setAttribute('data-height','20');pl.appendChild(pi);b.appendChild(pl);}}const inlineTarget=node.querySelector('.csl-right-inline')||node;inlineTarget.appendChild(b);});if(typeof _altmetric_embed_init==='function')_altmetric_embed_init();if(window.__dimensions_embed&&window.__dimensions_embed.addBadges)window.__dimensions_embed.addBadges();if(window.Iconify&&window.Iconify.scan)window.Iconify.scan();}if(document.readyState==='loading'){document.addEventListener('DOMContentLoaded',e);}else{e();}})();</script>
```"""

def matches_author(creators, search_last, search_initials):
    """
    Check if any creator matches the given last name and initials.
    """
    for creator in creators:
        last = creator.get('lastName', '').lower()
        first = creator.get('firstName', '').lower()
        if not first:
            continue
        
        first_initials = ''.join([part[0] for part in first.split() if part])
        
        if last == search_last.lower() and first_initials.upper().startswith(search_initials.upper()):
            return True
    return False

def get_role_categories(row):
    role_columns = [
        'Ultimate Degree-Role',
        'Penultimate Degree-Role',
        'Antepenultimate Degree-Role',
        'Preantepenultimate Degree-Role',
        'Propreantepenultimate Degree-Role',
        'Ultrasuprapropreantepenultimate Degree-Role',
    ]

    categories = []
    seen = set()
    for col in role_columns:
        value = row.get(col, None)
        if pd.isnull(value):
            continue

        text = str(value).strip()
        if not text or text.lower() in {'nan', 'nat', 'none'}:
            continue

        if text not in seen:
            categories.append(text)
            seen.add(text)

    if not categories:
        categories.append('Uncategorized')

    return categories

In [3]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Prefer safer, broader crops: center a credible subject face when available, otherwise keep a wide square crop.
HEADSHOT_FACE_HEIGHT_TARGET_RATIO = 0.35
HEADSHOT_FACE_HEIGHT_MIN_RATIO = 0.24
HEADSHOT_FACE_MIN_HEIGHT_RATIO = 0.10
HEADSHOT_MIN_CROP_SCALE_WITH_FACE = 0.82
HEADSHOT_FACE_EYE_LINE_RATIO = 0.40
HEADSHOT_FACE_CENTER_TARGET_X = 0.5
HEADSHOT_FACE_CENTER_TARGET_Y = 0.40
HEADSHOT_FACE_DETECTION_MAX_DIM_PX = 1600
HEADSHOT_FALLBACK_CROP_CENTER_Y = 0.45
HEADSHOT_FALLBACK_CROP_SCALE = 1.0

def _face_detection_score(face, image_width, image_height):
    x, y, width, height = face
    face_area = width * height
    face_height_ratio = height / float(max(1, min(image_width, image_height)))

    center_x = (x + (width / 2.0)) / float(max(1, image_width))
    center_y = (y + (height / 2.0)) / float(max(1, image_height))
    center_distance = (
        ((center_x - HEADSHOT_FACE_CENTER_TARGET_X) / 0.45) ** 2
        + ((center_y - HEADSHOT_FACE_CENTER_TARGET_Y) / 0.50) ** 2
    ) ** 0.5
    center_score = 1.0 - min(1.0, center_distance)
    size_score = _clamp((face_height_ratio - 0.08) / 0.30)

    return face_area * max(0.2, center_score) ** 2 * (0.5 + size_score)

def _detect_face_box_opencv(img):
    if not HEADSHOT_ENABLE_FACE_DETECTION:
        return None

    try:
        import cv2
    except Exception:
        return None

    try:
        rgb = np.array(img.convert('RGB'))
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

        detection_scale = 1.0
        if max(gray.shape[:2]) > HEADSHOT_FACE_DETECTION_MAX_DIM_PX:
            detection_scale = HEADSHOT_FACE_DETECTION_MAX_DIM_PX / float(max(gray.shape[:2]))
            gray = cv2.resize(
                gray,
                (
                    max(1, int(round(gray.shape[1] * detection_scale))),
                    max(1, int(round(gray.shape[0] * detection_scale))),
                ),
                interpolation=cv2.INTER_AREA,
            )

        cascade_dir = getattr(getattr(cv2, 'data', None), 'haarcascades', '')
        if not cascade_dir:
            return None

        min_detect_side = min(gray.shape[:2])
        cascade_names = (
            'haarcascade_frontalface_default.xml',
            'haarcascade_frontalface_alt.xml',
            'haarcascade_frontalface_alt2.xml',
        )
        detection_passes = (
            (1.05, 4, max(24, min_detect_side // 32)),
            (1.03, 5, max(28, min_detect_side // 28)),
            (1.1, 6, max(40, min_detect_side // 20)),
        )

        best_face = None
        best_score = -1.0
        for cascade_name in cascade_names:
            detector = cv2.CascadeClassifier(str(Path(cascade_dir) / cascade_name))
            if detector.empty():
                continue

            for scale_factor, min_neighbors, min_side in detection_passes:
                faces = detector.detectMultiScale(
                    gray,
                    scaleFactor=scale_factor,
                    minNeighbors=min_neighbors,
                    minSize=(min_side, min_side),
                )

                for face in faces:
                    face = tuple(int(value) for value in face)
                    score = _face_detection_score(face, gray.shape[1], gray.shape[0])
                    if score > best_score:
                        best_face = face
                        best_score = score

        if best_face is None:
            return None

        x, y, width, height = best_face
        if detection_scale != 1.0:
            x = int(round(x / detection_scale))
            y = int(round(y / detection_scale))
            width = int(round(width / detection_scale))
            height = int(round(height / detection_scale))

        face_height_ratio = height / float(max(1, min(img.width, img.height)))
        if face_height_ratio < HEADSHOT_FACE_MIN_HEIGHT_RATIO:
            return None

        return (x, y, width, height)
    except Exception:
        return None

def _get_headshot_crop_box(img, face_box=None):
    min_side = min(img.width, img.height)
    if min_side <= 0:
        return (0, 0, img.width, img.height)

    crop_side = min_side
    if face_box is None:
        face_box = _detect_face_box_opencv(img)
    if face_box:
        x, y, width, height = face_box
        face_center_x = x + (width / 2.0)
        face_eye_y = y + (height * HEADSHOT_FACE_EYE_LINE_RATIO)

        target_side = height / HEADSHOT_FACE_HEIGHT_TARGET_RATIO
        max_side_from_face = height / HEADSHOT_FACE_HEIGHT_MIN_RATIO
        face_based_side = int(round(max(height + 2, min(target_side, max_side_from_face))))
        min_face_crop_side = int(round(min_side * HEADSHOT_MIN_CROP_SCALE_WITH_FACE))
        crop_side = max(min_face_crop_side, face_based_side)
        crop_side = max(1, min(crop_side, min_side))

        left = round(face_center_x - (crop_side * HEADSHOT_FACE_CENTER_TARGET_X))
        top = round(face_eye_y - (crop_side * HEADSHOT_FACE_CENTER_TARGET_Y))
    else:
        crop_side = max(1, int(round(min_side * HEADSHOT_FALLBACK_CROP_SCALE)))
        left = round((img.width - crop_side) * HEADSHOT_CROP_CENTER_X)
        top = round((img.height - crop_side) * HEADSHOT_FALLBACK_CROP_CENTER_Y)

    left = int(max(0, min(left, img.width - crop_side)))
    top = int(max(0, min(top, img.height - crop_side)))
    return (left, top, left + crop_side, top + crop_side)

In [4]:
#| output: false
#| eval: true
#| include: false
#| context: setup

HEADSHOT_FACE_HEIGHT_TARGET_RATIO = 0.30
HEADSHOT_FACE_HEIGHT_MIN_RATIO = 0.22
HEADSHOT_FACE_MIN_HEIGHT_RATIO = 0.10
HEADSHOT_FACE_CROP_SCALE = 0.92
HEADSHOT_FALLBACK_CROP_SCALE = 1.0
HEADSHOT_FALLBACK_CROP_CENTER_Y = 0.50


def _pad_to_square_no_zoom(img):
    if img.width == img.height:
        return img

    square_side = max(img.width, img.height)
    padded = Image.new('RGB', (square_side, square_side), (255, 255, 255))
    source = img.convert('RGB') if img.mode != 'RGB' else img
    left = (square_side - source.width) // 2
    top = (square_side - source.height) // 2
    padded.paste(source, (left, top))
    return padded


def _get_headshot_crop_box(img, face_box=None):
    min_side = min(img.width, img.height)
    if min_side <= 0:
        return (0, 0, img.width, img.height)

    if face_box is None:
        face_box = _detect_face_box_opencv(img)
    if not face_box:
        return None

    x, y, width, height = face_box
    face_center_x = x + (width / 2.0)
    face_eye_y = y + (height * HEADSHOT_FACE_EYE_LINE_RATIO)

    crop_side = max(1, int(round(min_side * HEADSHOT_FACE_CROP_SCALE)))
    crop_side = min(crop_side, min_side)

    left = round(face_center_x - (crop_side / 2.0))
    top = round(face_eye_y - (crop_side * HEADSHOT_FACE_CENTER_TARGET_Y))
    left = int(max(0, min(left, img.width - crop_side)))
    top = int(max(0, min(top, img.height - crop_side)))
    return (left, top, left + crop_side, top + crop_side)


def copy_downsampled_headshot(private_path, public_path):
    src = Path(private_path)
    dst = Path(public_path)
    dst.parent.mkdir(parents=True, exist_ok=True)

    needs_sync = not dst.exists()
    if not needs_sync:
        try:
            with Image.open(dst) as existing_img:
                needs_sync = existing_img.width != existing_img.height or existing_img.width > HEADSHOT_MAX_DIM_PX or existing_img.height > HEADSHOT_MAX_DIM_PX
        except Exception:
            needs_sync = True

    if not needs_sync and HEADSHOT_SOURCE_NOTEBOOK_PATH.exists():
        needs_sync = HEADSHOT_SOURCE_NOTEBOOK_PATH.stat().st_mtime > dst.stat().st_mtime
    if not needs_sync:
        return

    try:
        with Image.open(src) as img:
            exif = img.getexif()
            img = ImageOps.exif_transpose(img)

            try:
                del exif[274]
            except Exception:
                pass

            face_box = _detect_face_box_opencv(img)
            if face_box is None:
                img = _pad_to_square_no_zoom(img)
            else:
                crop_box = _get_headshot_crop_box(img, face_box=face_box)
                img = img.crop(crop_box)

            if img.width > HEADSHOT_MAX_DIM_PX or img.height > HEADSHOT_MAX_DIM_PX:
                img.thumbnail((HEADSHOT_MAX_DIM_PX, HEADSHOT_MAX_DIM_PX), Image.Resampling.LANCZOS)

            suffix = dst.suffix.lower()
            save_kwargs = {'dpi': (HEADSHOT_WEB_DPI, HEADSHOT_WEB_DPI)}

            if suffix in ('.jpg', '.jpeg'):
                if img.mode not in ('RGB', 'L'):
                    img = img.convert('RGB')
                save_kwargs.update({'quality': 85, 'optimize': True, 'progressive': True, 'exif': exif.tobytes()})
            elif suffix == '.png':
                save_kwargs.update({'optimize': True, 'compress_level': 9})

            img.save(dst, **save_kwargs)
    except Exception as e:
        shutil.copy2(src, dst)
        print(f"Warning: copied without downsampling for {src.name}: {e}")

In [5]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Generate cards for current group members
import os

citation_format = "ieee"

def save_with_dirs(path, array, **kwargs):
    """
    Save a NumPy array to a file, creating directories if needed.

    Parameters
    ----------
    path : str
        Full path (including filename) where the array will be saved.
    array : np.ndarray
        The data to save.
    **kwargs
        Additional arguments passed to numpy.savetxt.
    """
    # Get directory part of the path
    dir_path = os.path.dirname(path)

    if dir_path:  # Only try to create if there's a directory specified
        os.makedirs(dir_path, exist_ok=True)

    # Save the array
    np.savetxt(path, array, **kwargs)

for index, row in df[df['current']==True].iterrows():

   linkedin = ''
   orcid = ''
   github =''
   googlescholar = ''
   categories = get_role_categories(row)
   categories_yaml = ', '.join([f'"{item}"' for item in categories])
   txt = "".join(['---\n'+\
               'title: "{0}"\n'+\
               'categories: [{1}]\n'+\
               'member-since: {2}\n'+\
               'date: today\n'+\
               'date-modified: {2}\n'+\
               'date-format: "MMMM YYYY"\n'+\
               'language:\n'+\
               '  title-block-published: "Updated"\n'+\
               '  title-block-modified: "Joined"\n'+\
               'execute:\n' +\
               '  echo: false\n' +\
               'image: ']).format(row['Display Name'],
                                  categories_yaml, 
                                  row["Ultimate Role Start"])
   
   member_slug = row['Display Name'].replace(' ', '_')
   private_headshot_path = "private/Group Member Photos/{0}.jpg".format(member_slug)
   public_headshot_path = "files/photos/People/{0}.jpg".format(member_slug)
   headshot = "../../{0}".format(public_headshot_path)
   placeholder = "../../files/photos/People/{0}.jpg".format('anon')
   if os.path.exists(private_headshot_path):
      copy_downsampled_headshot(private_headshot_path, public_headshot_path)
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   elif os.path.exists(public_headshot_path):
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   else:
      txt += "{0}\n".format(placeholder)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Stock photo of a dog wearing glasses."\n']).format(placeholder)

   txt += "  image-shape: round\n"
   links = []

   if not pd.isnull(row['LinkedIn']):
      links.append(
         '    - text: "{{{{< iconify mdi linkedin >}}}}"\n'
         '      url: https://www.linkedin.com/in/{0}\n'.format(row['LinkedIn'])
      )
   if not pd.isnull(row['ORCID']):
      links.append(
         '    - text: "{{{{< iconify simple-icons orcid >}}}}"\n'
         '      url: https://orcid.org/{0}\n'.format(row['ORCID'])
      )
   if not pd.isnull(row['Google Scholar']):
      links.append(
         '    - text: "{{{{< iconify academicons google-scholar >}}}}"\n'
         '      url: https://scholar.google.com/citations?user={0}&hl=en\n'.format(row['Google Scholar'])
      )
   if not pd.isnull(row['GitHub']):
      links.append(
         '    - text: "{{{{< iconify mdi github >}}}}"\n'
         '      url: https://github.com/{0}\n'.format(row['GitHub'])
      )

   state_codes = []
   country_codes = []
   for i in range(1,10):
      col_name = "Hometown {0}".format(i)
      if not pd.isnull(row[col_name]):
         code = get_country_iso_code_nominatim(row[col_name])
         country_codes.append(code)
         
         if code == "US":
               state_codes.append(get_us_state_code(row[col_name]))
    
   # Store only unique values
   country_codes = list(set(
      [item for item in country_codes if item is not None]))
   state_codes = list(set(
      [item for item in state_codes if item is not None]))

   flags = []
   for item in country_codes:
      url = get_country_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_wikipedia_url_from_country_code(item))
      )
   for item in state_codes:
      url = get_us_state_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_us_state_wikipedia_url(item))
      )

   if links:
      txt += "  links:\n"
      txt += ''.join(links)

   txt += '\n---\n\n'

   people_workproducts_assets = get_people_workproducts_assets()
   publications = ''

   if pd.isnull(row['Authorship Name']):
      nom = row['Display Name'].split(' ')[-1]
   else:
      nom = row['Authorship Name'].split('.')[0:1][0]

   results = get_zotero_items_by_author_and_type(
      nom, ['thesis'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Thesis/Dissertation\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['journalArticle', 'book', 'bookSection', 'conferencePaper', 'report'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Publications\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['presentation'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      presenter_html, contributor_html = get_presentation_citations_all_authors(results, row['Display Name'])
      if presenter_html:
         publications += '# Presentations\n\n' + \
         "```{=html}\n" +\
         presenter_html + '\n```\n\n'
      if contributor_html:
         publications += '# Contributions\n\n' + \
         "```{=html}\n" +\
         contributor_html + '\n```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['computerProgram'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Code & Datasets\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'
   
   publication = ''.join(['*CARD Lab Work Products:*\n', publications])

   x = np.array(['\n'.join([txt, people_workproducts_assets, publications])])
   filepath = ''.join(['./people/current/', row['Display Name'].replace(' ', '_'), '.qmd'])
   save_with_dirs(filepath, x, fmt='%s')


/var/folders/nz/l5ynytfs6sj2drjd5cs1wckc0000gn/T/ipykernel_85874/3147098979.py:509: XMLParsedAsHTMLWarning:

It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)




In [6]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Python cell to generate a listing .qmd file
listing_qmd_path = "people/current.qmd"

listing_yaml = """
---
title: "Current Group Members"
format: html
listing:
  type: grid
  contents: current
  template: _partials/people-listing.ejs.md
  sort: "date asc"
  image-placeholder: files/images/anon.jpg
  date-format: "MMM YYYY"
  categories: numbered
  image-height: 250px
  grid-columns: 4
  max-items: 100 # Show all items
  filter-ui: [categories, date, title]
include-in-header:
  text: |
      <style>
      /* Dashboard grid listing tweaks! /*

      /* Hide pop-out / maximize controls for listing items */
      .quarto-listing .listing-item .card-header-actions {
      display: none !important;
      }

      .quarto-listing .listing-item .card-header {
      cursor: default;
      }
      </style>
---
"""

# Write the YAML to the .qmd file
with open(listing_qmd_path, "w") as f:
    f.write(listing_yaml)

print(f"Listing page written to {listing_qmd_path}")

Listing page written to people/current.qmd


In [7]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Generate cards for group alumni
import os

citation_format = "ieee"

def save_with_dirs(path, array, **kwargs):
    """
    Save a NumPy array to a file, creating directories if needed.

    Parameters
    ----------
    path : str
        Full path (including filename) where the array will be saved.
    array : np.ndarray
        The data to save.
    **kwargs
        Additional arguments passed to numpy.savetxt.
    """
    # Get directory part of the path
    dir_path = os.path.dirname(path)

    if dir_path:  # Only try to create if there's a directory specified
        os.makedirs(dir_path, exist_ok=True)

    # Save the array
    np.savetxt(path, array, **kwargs)

def format_member_date(value):
    if pd.isnull(value):
        return ''
    return pd.to_datetime(value).strftime('%B %Y')

for index, row in df[df['current']==False].iterrows():

   linkedin = ''
   orcid = ''
   github =''
   googlescholar = ''
   categories = get_role_categories(row)
   categories_yaml = ', '.join([f'"{item}"' for item in categories])
   member_range = '{0} to {1}'.format(format_member_date(row["Ultimate Role Start"]), format_member_date(row["Ultimate Role Finish"]))
   txt = "".join(['---\n'+\
               'title: "{0}"\n'+\
               'categories: [{1}]\n'+\
               'member-from: {2}\n'+\
               'member-to: {3}\n'+\
               'date: today\n'+\
               'date-modified: {2}\n'+\
               'date-format: "MMM YYYY"\n'+\
               'language:\n'+\
               '  title-block-published: "Updated"\n'+\
               '  title-block-modified: "Member from"\n'+\
               'execute:\n' +\
               '  echo: false\n' +\
               'image: ']).format(row['Display Name'],
                                  categories_yaml, 
                                  row["Ultimate Role Start"],
                                  row["Ultimate Role Finish"])
   
   member_slug = row['Display Name'].replace(' ', '_')
   private_headshot_path = "private/Group Member Photos/{0}.jpg".format(member_slug)
   public_headshot_path = "files/photos/People/{0}.jpg".format(member_slug)
   headshot = "../../{0}".format(public_headshot_path)
   placeholder = "../../files/photos/People/{0}.jpg".format('anon')
   if os.path.exists(private_headshot_path):
      copy_downsampled_headshot(private_headshot_path, public_headshot_path)
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   elif os.path.exists(public_headshot_path):
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   else:
      txt += "{0}\n".format(placeholder)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Stock photo of a dog wearing glasses."\n']).format(placeholder)

   txt += "  image-shape: round\n"
   links = []

   if not pd.isnull(row['LinkedIn']):
      links.append(
         '    - text: "{{{{< iconify mdi linkedin >}}}}"\n'
         '      url: https://www.linkedin.com/in/{0}\n'.format(row['LinkedIn'])
      )
   if not pd.isnull(row['ORCID']):
      links.append(
         '    - text: "{{{{< iconify simple-icons orcid >}}}}"\n'
         '      url: https://orcid.org/{0}\n'.format(row['ORCID'])
      )
   if not pd.isnull(row['Google Scholar']):
      links.append(
         '    - text: "{{{{< iconify academicons google-scholar >}}}}"\n'
         '      url: https://scholar.google.com/citations?user={0}&hl=en\n'.format(row['Google Scholar'])
      )
   if not pd.isnull(row['GitHub']):
      links.append(
         '    - text: "{{{{< iconify mdi github >}}}}"\n'
         '      url: https://github.com/{0}\n'.format(row['GitHub'])
      )

   state_codes = []
   country_codes = []
   for i in range(1,10):
      col_name = "Hometown {0}".format(i)
      if not pd.isnull(row[col_name]):
         code = get_country_iso_code_nominatim(row[col_name])
         country_codes.append(code)
         
         if code == "US":
               state_codes.append(get_us_state_code(row[col_name]))
    
   # Store only unique values
   country_codes = list(set(
      [item for item in country_codes if item is not None]))
   state_codes = list(set(
      [item for item in state_codes if item is not None]))

   flags = []
   for item in country_codes:
      url = get_country_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_wikipedia_url_from_country_code(item))
      )
   for item in state_codes:
      url = get_us_state_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_us_state_wikipedia_url(item))
      )

   if links:
      txt += "  links:\n"
      txt += ''.join(links)

   txt += '\n---\n\n'
   txt += ''.join([
      '```{=html}\n',
      '<div class="member-range-source" data-member-range="' + member_range + '"></div>\n',
      '<script>\n',
      'document.addEventListener("DOMContentLoaded", function () {\n',
      '  const source = document.querySelector(".member-range-source");\n',
      '  const modified = document.querySelector(".quarto-title-meta .date-modified");\n',
      '  if (source && modified) {\n',
      '    const range = source.dataset.memberRange || modified.textContent;\n',
      '    const contents = modified.closest(".quarto-title-meta-contents");\n',
      '    const heading = contents ? contents.previousElementSibling : null;\n',
      '    if (heading && heading.classList.contains("quarto-title-meta-heading")) {\n',
      '      heading.textContent = `Member from ${range}`;\n',
      '      if (contents) { contents.remove(); }\n',
      '    } else {\n',
      '      modified.textContent = range;\n',
      '    }\n',
      '  }\n',
      '});\n',
      '</script>\n',
      '```\n\n'])

   people_workproducts_assets = get_people_workproducts_assets()
   publications = ''

   if pd.isnull(row['Authorship Name']):
      nom = row['Display Name'].split(' ')[-1]
   else:
      nom = row['Authorship Name'].split('.')[0:1][0]

   results = get_zotero_items_by_author_and_type(
      nom, ['thesis'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Thesis/Dissertation\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['journalArticle', 'book', 'bookSection', 'conferencePaper', 'report'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Publications\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['presentation'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      presenter_html, contributor_html = get_presentation_citations_all_authors(results, row['Display Name'])
      if presenter_html:
         publications += '# Presentations\n\n' + \
         "```{=html}\n" +\
         presenter_html + '\n```\n\n'
      if contributor_html:
         publications += '# Contributions\n\n' + \
         "```{=html}\n" +\
         contributor_html + '\n```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['computerProgram'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Code & Datasets\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'
   
   publication = ''.join(['*CARD Lab Work Products:*\n', publications])

   x = np.array(['\n'.join([txt, people_workproducts_assets, publications])])
   filepath = ''.join(['./people/alumni/', row['Display Name'].replace(' ', '_'), '.qmd'])
   save_with_dirs(filepath, x, fmt='%s')

/var/folders/nz/l5ynytfs6sj2drjd5cs1wckc0000gn/T/ipykernel_85874/3147098979.py:509: XMLParsedAsHTMLWarning:

It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)




In [8]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Python cell to generate a listing .qmd file
listing_qmd_path = "people/alumni.qmd"

listing_yaml = """
---
title: "Alumni"
format: html
listing:
  type: grid
  contents: alumni
  template: _partials/people-listing.ejs.md
  sort: "member-to desc"
  image-placeholder: files/images/anon.jpg
  date-format: "MMM YYYY"
  image-height: 250px
  grid-columns: 4
  page-size: 100
  categories: numbered
  filter-ui: [categories, date, title]
include-in-header:
  text: |
      <style>
      /* Hide pop-out / maximize controls for listing items */
      .quarto-listing .listing-item .card-header-actions {
      display: none !important;
      }

      .quarto-listing .listing-item .card-header {
      cursor: default;
      }
      </style>
---
"""

# Write the YAML to the .qmd file
with open(listing_qmd_path, "w") as f:
    f.write(listing_yaml)

print(f"Listing page written to {listing_qmd_path}")

Listing page written to people/alumni.qmd


In [9]:
#| output: true
#| eval: true
#| include: false
#| context: setup

import subprocess, sys
result = subprocess.run(
    [sys.executable, "scripts/recycle-stale-people.py"],
    capture_output=True, text=True
)
if result.stdout:
    print(result.stdout)
if result.returncode != 0 and result.stderr:
    print(result.stderr, file=sys.stderr)


[recycle-stale-people] Checking for stale profile files...
  No stale current profiles found.
  No stale alumni profiles found.
[recycle-stale-people] Done. Total recycled: 0

